# Día 10 · Unidad 12 — Cópulas: modelar dependencia entre líneas de negocio

**Objetivos de aprendizaje**
- Construir una cópula gaussiana desde cero, sin librerías especializadas: normales correlacionadas → uniformes → marginales elegidas.
- Simular severidades correlacionadas de dos ramos de negocio y comprobar que la correlación simulada reproduce la correlación objetivo.
- Distinguir la correlación lineal (Pearson) de la estructura de dependencia que aporta una cópula.

**Duración estimada:** 45-60 minutos.

**Requisitos previos:** `05_teoria_opciones_montecarlo.ipynb` de este mismo día. Se asume dominio previo del concepto de cópula — aquí el foco es exclusivamente la implementación.

## 0. Por qué a mano, sin librería de cópulas

`requirements.txt` no incluye ninguna librería especializada en cópulas (`copulas`, `statsmodels` con sus módulos de cópulas, etc.) — no hace falta: una cópula gaussiana se construye con tres pasos que ya sabéis hacer con NumPy/SciPy:

1. Generar variables normales estándar **correlacionadas** según una matriz de correlación objetivo (`numpy.random.multivariate_normal`).
2. Transformarlas a uniformes $(0,1)$ aplicando la función de distribución acumulada de la normal a cada una (`scipy.stats.norm.cdf`) — esto es lo que se llama la *transformada integral de probabilidad*.
3. Transformar cada uniforme a la marginal que se quiera (aquí, severidades de siniestro) aplicando la inversa de su función de distribución (`.ppf()` de la marginal elegida).

La cópula gaussiana es precisamente la estructura de dependencia implícita en el paso 1: las marginales del paso 3 pueden ser completamente distintas entre sí (una lognormal y una gamma, por ejemplo) y seguir compartiendo esa misma estructura de dependencia.

## 1. Motivación con datos del curso: severidad de AUTO y HOGAR

Antes de construir el ejemplo con parámetros inventados, miramos brevemente si hay indicios de relación entre los importes de siniestros de dos ramos en `data/raw/siniestros.csv` — agregando por mes, para tener una única serie comparable entre ramos (a nivel de siniestro individual no hay ninguna razón para que un siniestro de AUTO y uno de HOGAR estén "emparejados").

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm, lognorm

siniestros = pd.read_csv("../data/raw/siniestros.csv", parse_dates=["fecha_siniestro"])

siniestros["anio_mes"] = siniestros["fecha_siniestro"].dt.to_period("M")
importes_mensuales = (
    siniestros[siniestros["producto"].isin(["AUTO", "HOGAR"])]
    .pivot_table(index="anio_mes", columns="producto", values="importe_pagado", aggfunc="sum")
    .fillna(0.0)
)
correlacion_observada = importes_mensuales["AUTO"].corr(importes_mensuales["HOGAR"])
print(f"Correlacion (Pearson) del importe mensual pagado, AUTO vs HOGAR: {correlacion_observada:.3f}")
print(f"Meses observados: {len(importes_mensuales)}")

Correlacion (Pearson) del importe mensual pagado, AUTO vs HOGAR: 0.177
Meses observados: 78


Con pocos meses de datos, esta correlación observada es solo orientativa (no es una estimación fiable de una correlación poblacional) — la usamos únicamente como **gancho** de motivación, no como el parámetro que vamos a usar en el ejemplo. Para el ejemplo de cópula usamos un parámetro de correlación inventado pero razonable ($\rho = 0.5$, dependencia moderada-alta entre ramos, coherente con exposiciones que comparten factores comunes como climatología o ciclo económico regional), documentado como tal.

## 2. Cópula gaussiana paso a paso

### Paso 1: normales correlacionadas

In [2]:
rho = 0.5  # correlacion objetivo entre AUTO y HOGAR (parametro elegido, no estimado)
n_sims = 50_000

matriz_correlacion = np.array([
    [1.0, rho],
    [rho, 1.0],
])

rng = np.random.default_rng(42)
normales_correlacionadas = rng.multivariate_normal(mean=[0.0, 0.0], cov=matriz_correlacion, size=n_sims)

correlacion_simulada = np.corrcoef(normales_correlacionadas[:, 0], normales_correlacionadas[:, 1])[0, 1]
print(f"Correlacion objetivo:  {rho}")
print(f"Correlacion simulada (normales): {correlacion_simulada:.4f}")
assert abs(correlacion_simulada - rho) < 0.01
print("La correlacion simulada reproduce la correlacion objetivo.")

Correlacion objetivo:  0.5
Correlacion simulada (normales): 0.4926
La correlacion simulada reproduce la correlacion objetivo.


### Paso 2: transformar a uniformes (transformada integral de probabilidad)

In [3]:
uniformes = norm.cdf(normales_correlacionadas)  # cada columna, marginalmente, es U(0,1)
u_auto, u_hogar = uniformes[:, 0], uniformes[:, 1]

print(f"Rango de u_auto:  [{u_auto.min():.4f}, {u_auto.max():.4f}]")
print(f"Media de u_auto (deberia rondar 0.5):  {u_auto.mean():.4f}")
print(f"Correlacion de RANGOS (Spearman) entre u_auto y u_hogar: {pd.Series(u_auto).corr(pd.Series(u_hogar), method='spearman'):.4f}")

Rango de u_auto:  [0.0000, 1.0000]
Media de u_auto (deberia rondar 0.5):  0.5021
Correlacion de RANGOS (Spearman) entre u_auto y u_hogar: 0.4733


> ⚠️ **Error típico**: la correlación de Pearson entre las **normales** ($\rho$) no es exactamente igual a la correlación de Pearson entre las **marginales finales** una vez transformadas al paso 3 (a no ser que las marginales también sean normales) — la cópula gaussiana preserva la correlación de **rangos** (Spearman), no necesariamente la correlación lineal exacta tras una transformación no lineal como la de una lognormal. Es un matiz importante: "correlación 0.5 en la cópula" no significa automáticamente "correlación de Pearson 0.5 en las severidades finales".

### Paso 3: transformar a las marginales elegidas

In [4]:
# Severidad de AUTO: lognormal con media ~2.200 EUR, cola pesada
# Severidad de HOGAR: lognormal con media ~900 EUR, mas concentrada
parametros_auto = {"s": 0.9, "scale": np.exp(7.4)}    # s = sigma de la lognormal, scale = exp(mu)
parametros_hogar = {"s": 0.6, "scale": np.exp(6.6)}

severidad_auto = lognorm.ppf(u_auto, **parametros_auto)
severidad_hogar = lognorm.ppf(u_hogar, **parametros_hogar)

print(f"Severidad AUTO simulada:  media={severidad_auto.mean():,.2f}  mediana={np.median(severidad_auto):,.2f}")
print(f"Severidad HOGAR simulada: media={severidad_hogar.mean():,.2f}  mediana={np.median(severidad_hogar):,.2f}")

correlacion_pearson_final = np.corrcoef(severidad_auto, severidad_hogar)[0, 1]
correlacion_spearman_final = pd.Series(severidad_auto).corr(pd.Series(severidad_hogar), method="spearman")

# La correlacion de Spearman es invariante a transformaciones monotonas (como
# pasar de normal a lognormal), asi que coincide con la de los uniformes del
# paso 2 - pero NO coincide exactamente con rho: para una copula gaussiana,
# la relacion teorica exacta entre el rho de Pearson de las normales y el
# rho de Spearman resultante es rho_spearman = (6/pi) * arcsin(rho/2).
rho_spearman_teorico = (6 / np.pi) * np.arcsin(rho / 2)

print(f"\nCorrelacion de Pearson entre severidades finales:  {correlacion_pearson_final:.4f}")
print(f"Correlacion de Spearman entre severidades finales: {correlacion_spearman_final:.4f}")
print(f"Spearman teorico para rho={rho}: (6/pi)*arcsin(rho/2) = {rho_spearman_teorico:.4f}")
assert abs(correlacion_spearman_final - rho_spearman_teorico) < 0.02
print("\nLa correlacion de rangos (Spearman) simulada coincide con la formula teorica de la copula gaussiana.")

Severidad AUTO simulada:  media=2,463.76  mediana=1,648.15
Severidad HOGAR simulada: media=882.08  mediana=736.40

Correlacion de Pearson entre severidades finales:  0.4159
Correlacion de Spearman entre severidades finales: 0.4733
Spearman teorico para rho=0.5: (6/pi)*arcsin(rho/2) = 0.4826

La correlacion de rangos (Spearman) simulada coincide con la formula teorica de la copula gaussiana.


Fíjate en la diferencia entre las tres líneas de output: la correlación de **Spearman** de las severidades finales es invariante a transformaciones monótonas (pasar de normal a lognormal no la cambia), pero **no coincide exactamente con $\rho$** — coincide con $(6/\pi)\arcsin(\rho/2)$, la relación teórica exacta entre el $\rho$ de Pearson de las normales subyacentes y el $\rho$ de Spearman que produce una cópula gaussiana (para $\rho=0.5$, da $\approx 0.483$, muy cerca de $\rho$ pero no igual). La correlación de **Pearson** de las severidades finales es, además, distinta de ambas — más baja, porque la transformación a lognormal comprime/expande de forma no lineal.

Esta es la razón práctica por la que, al calibrar una cópula gaussiana a partir de datos observados, se suele calibrar $\rho$ invirtiendo esa fórmula a partir de la correlación de **rangos** de los datos (Spearman o Kendall), no a partir de su correlación de Pearson directamente.

## 3. Caso de control: independencia ($\rho = 0$)

Antes de confiar en la construcción, comprobamos el caso trivial: con $\rho = 0$ las dos marginales deben salir prácticamente independientes.

In [5]:
matriz_independencia = np.eye(2)
normales_independientes = rng.multivariate_normal(mean=[0.0, 0.0], cov=matriz_independencia, size=n_sims)
u_auto_indep, u_hogar_indep = norm.cdf(normales_independientes[:, 0]), norm.cdf(normales_independientes[:, 1])

severidad_auto_indep = lognorm.ppf(u_auto_indep, **parametros_auto)
severidad_hogar_indep = lognorm.ppf(u_hogar_indep, **parametros_hogar)

correlacion_bajo_independencia = np.corrcoef(severidad_auto_indep, severidad_hogar_indep)[0, 1]
print(f"Correlacion con rho=0 (deberia rondar 0): {correlacion_bajo_independencia:.4f}")
assert abs(correlacion_bajo_independencia) < 0.02
print("Correcto: sin correlacion en la copula, las severidades finales salen practicamente incorreladas.")

Correlacion con rho=0 (deberia rondar 0): -0.0037
Correcto: sin correlacion en la copula, las severidades finales salen practicamente incorreladas.


## 4. Aplicación: impacto de la dependencia en la severidad agregada

El motivo por el que esto importa en la práctica: si se ignora la correlación entre ramos (se asume independencia cuando en realidad hay dependencia positiva), se **subestima** la probabilidad de escenarios adversos simultáneos en ambos ramos — relevante para capital de solvencia, reaseguro combinado, o agregados de siniestralidad de una cartera multirriesgo.

In [6]:
severidad_conjunta_dependiente = severidad_auto + severidad_hogar
severidad_conjunta_independiente = severidad_auto_indep + severidad_hogar_indep

percentil = 99.5  # tipico de un calculo de capital de solvencia
print(f"Percentil {percentil} de la severidad conjunta:")
print(f"  con dependencia (rho={rho}):  {np.percentile(severidad_conjunta_dependiente, percentil):,.2f} EUR")
print(f"  asumiendo independencia:      {np.percentile(severidad_conjunta_independiente, percentil):,.2f} EUR")

Percentil 99.5 de la severidad conjunta:
  con dependencia (rho=0.5):  18,186.85 EUR
  asumiendo independencia:      18,010.84 EUR


El percentil extremo de la suma es más alto cuando se modela la dependencia real entre ramos: asumir independencia por conveniencia de cálculo, cuando en realidad hay correlación positiva, infravalora el riesgo de cola — exactamente el tipo de supuesto que, según `docs/uso_ia_en_el_curso.md`, una IA puede ayudar a **implementar** una vez decidido el método, pero nunca a **decidir** por vosotros si es razonable para vuestra cartera.

**Para ti:** repite la sección 2 con $\rho = 0.9$ (dependencia muy alta) y compara el percentil 99.5 de la severidad conjunta con el obtenido arriba para $\rho = 0.5$.

**Para ti:** añade un tercer ramo (`SALUD`) a la cópula, con su propia marginal lognormal y su propia fila/columna en `matriz_correlacion` (una matriz 3x3). Pista: sigue siendo una única llamada a `rng.multivariate_normal` con `mean=[0, 0, 0]`.

## Resumen

- Una cópula gaussiana se construye en tres pasos con NumPy/SciPy, sin librería especializada: normales correlacionadas (`multivariate_normal`) → uniformes (`norm.cdf`) → marginales elegidas (`.ppf()` de la distribución que se quiera).
- Las marginales finales pueden ser de familias completamente distintas entre sí; lo que la cópula fija es la estructura de dependencia, no la forma de cada marginal.
- La cópula gaussiana preserva la correlación de **rangos** (Spearman) igual al parámetro $\rho$ usado; la correlación de Pearson de las marginales finales, tras una transformación no lineal, es distinta — hay que calibrar y validar contra la métrica correcta.
- El caso de control $\rho = 0$ (independencia) es la primera validación obligatoria de cualquier implementación de cópula, antes de confiar en los casos con dependencia.
- Ignorar la dependencia entre líneas de negocio cuando existe de verdad subestima sistemáticamente el riesgo de escenarios adversos simultáneos — con implicaciones directas en capital y reaseguro.

**Siguiente:** `07_teoria_provisiones_chain_ladder.ipynb` — el método Chain Ladder para el cálculo de provisiones.